# Generate Synthetic Grade-5 Math CoT Data

In [1]:
# 1. Install dependencies
!pip install transformers accelerate bitsandbytes nbformat --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 48.0 MB/s eta 0:00:00


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
import json
import re

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

MODEL_NAME = "tiiuae/falcon-7b-instruct"

# Load tokenizer & 4-bit model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Build text-generation pipeline with increased token limit
gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,  # Increased for more questions per batch
    do_sample=True,
    top_p=0.9,
    temperature=0.7,
)

tokenizer_config.json:   0%|          | 0.00/1.13k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/281 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

configuration_falcon.py:   0%|          | 0.00/7.16k [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/9.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.48G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

Device set to use cuda:0


In [4]:
# Function to generate questions in batches
def generate_questions_batch(batch_size=10):
    """Generate a batch of questions"""
    prompt = (
        "You are a friendly tutor. Generate exactly "
        f"{batch_size} simple Grade-5 math questions. "
        "Cover addition, subtraction, multiplication, division, and rectangle perimeter/area. "
        "Format each question on a new line starting with a number:\n"
        "1. [question]\n"
        "2. [question]\n"
        "etc."
    )

    out = gen(prompt)[0]["generated_text"][len(prompt):].strip()
    lines = [l.strip() for l in out.splitlines() if l.strip()]

    questions = []
    for line in lines:
        # Extract question after number and period
        match = re.match(r'^\d+\.\s*(.+)', line)
        if match:
            questions.append(match.group(1).strip())
        elif line and not re.match(r'^\d+\.$', line):  # Non-empty line without just number
            questions.append(line)

    return questions

def generate_questions(total_needed):
    """Generate total_needed questions by batching"""
    all_questions = []
    batch_size = 10  # Generate 10 questions at a time

    while len(all_questions) < total_needed:
        remaining = total_needed - len(all_questions)
        current_batch_size = min(batch_size, remaining)

        print(f"Generating batch of {current_batch_size} questions... ({len(all_questions)} completed)")

        batch_questions = generate_questions_batch(current_batch_size)

        # Filter out duplicates and add to main list
        for q in batch_questions:
            if q not in all_questions and len(all_questions) < total_needed:
                all_questions.append(q)

        # Safety break to avoid infinite loop
        if len(batch_questions) == 0:
            print("Warning: No questions generated in this batch. Stopping.")
            break

    return all_questions[:total_needed]  # Ensure exact count

def generate_cot(q):
    """Generate Chain of Thought reasoning for a question"""
    prompt = f"Question: {q}\nAnswer: Let's think step by step."
    out = gen(prompt)[0]["generated_text"][len(prompt):].strip()
    return out

In [5]:
# Generate & save JSONL
N = 200
print(f"Starting generation of {N} questions...")

questions = generate_questions(N)
print(f"Generated {len(questions)} unique questions")

# Generate CoT answers and save
with open("auto_grade5_math_cot.jsonl", "w") as fout:
    for i, q in enumerate(questions, 1):
        print(f"Generating CoT for question {i}/{len(questions)}")
        cot = generate_cot(q)
        entry = {
            "prompt": f"Question: {q}\nAnswer:",
            "completion": cot
        }
        fout.write(json.dumps(entry) + "\n")

print(f"Successfully wrote {len(questions)} Q&A pairs with CoT to auto_grade5_math_cot.jsonl")

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


Starting generation of 200 questions...
Generating batch of 10 questions... (0 completed)


Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


Generating batch of 10 questions... (9 completed)


Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


...

Generating CoT for question 198/200


Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


Generating CoT for question 199/200


Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


Generating CoT for question 200/200
Successfully wrote 200 Q&A pairs with CoT to auto_grade5_math_cot.jsonl


In [7]:
import json

data = []
with open('/content/auto_grade5_math_cot.jsonl', 'r') as f:
    for line in f:
        data.append(json.loads(line))

# Example: print the first entry
print(len(data))
print(data[199])

200
{'prompt': 'Question: Addition question:\nAnswer:', 'completion': "First, the 4 + 2 = 6. Then, the 6 + 3 = 9. So, 9 + 2 = 11. We can't add 2 and 11. We know that 11 is 2 more than 10, so we can't add 2 and 11. We also know that 11 is 2 more than 10, so we can't add 2 and 10 either. Therefore, 11 + 2 = 11."}
